In [1]:
import os
import csv
from pathlib import Path

#  Configuration
INPUT_FOLDER = "DATA_BULLETINS_CSV_INDIVIDUAL"  # Dossier avec les CSV individuels
OUTPUT_FOLDER = "DATA_BULLETINS_CSV_MERGED"  # Dossier pour les CSV fusionnés

# Créer le dossier de sortie
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

def merge_csv_files(category_path: str, category_name: str, output_file: str) -> bool:
    """Fusionne tous les fichiers CSV d'une catégorie"""
    print(f"\n Fusion: {category_name}")
    print("-" * 70)
    
    # Récupérer tous les fichiers CSV
    csv_files = sorted(Path(category_path).glob("*.csv"))
    
    if not csv_files:
        print(f"     Aucun fichier CSV trouvé")
        return False
    
    print(f"    {len(csv_files)} fichier(s) à fusionner")
    
    merged_header = None
    all_rows = []
    
    # Lire tous les fichiers CSV
    for idx, csv_file in enumerate(csv_files, 1):
        try:
            with open(csv_file, 'r', encoding='utf-8') as f:
                reader = csv.reader(f)
                rows = list(reader)
                
                if not rows:
                    continue
                
                # Premier fichier : définir l'en-tête
                if merged_header is None:
                    merged_header = rows[0]
                    print(f"   En-tête: {len(merged_header)} colonne(s)")
                
                # Ajouter les données (ignorer l'en-tête)
                data_rows = rows[1:]
                if data_rows:
                    all_rows.extend(data_rows)
                    print(f"   [{idx}/{len(csv_files)}] {csv_file.name}: {len(data_rows)} ligne(s)")
        
        except Exception as e:
            print(f"     Erreur lecture {csv_file.name}: {e}")
            continue
    
    if merged_header is None or not all_rows:
        print(f"    Aucune donnée à fusionner")
        return False
    
    # Écrire le fichier fusionné
    try:
        with open(output_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
            
            # Écrire l'en-tête
            writer.writerow(merged_header)
            
            # Écrire toutes les données
            for row in all_rows:
                # Normaliser le nombre de colonnes
                if len(row) < len(merged_header):
                    row.extend([''] * (len(merged_header) - len(row)))
                elif len(row) > len(merged_header):
                    row = row[:len(merged_header)]
                
                writer.writerow(row)
        
        print(f"    Fusionné: {len(all_rows)} lignes totales")
        print(f"    Fichier: {os.path.basename(output_file)}")
        return True
        
    except Exception as e:
        print(f"    Erreur écriture: {e}")
        return False

def merge_all_categories():
    """Fusionne tous les CSV par catégorie"""
    if not os.path.exists(INPUT_FOLDER):
        print(f" Dossier {INPUT_FOLDER} introuvable!")
        return
    
    print(" FUSION DES CSV PAR CATÉGORIE")
    print(f" Source: {os.path.abspath(INPUT_FOLDER)}")
    print(f" Sortie: {os.path.abspath(OUTPUT_FOLDER)}")
    print("=" * 70)
    
    categories_processed = 0
    categories_success = 0
    
    # Parcourir toutes les catégories
    for category in sorted(os.listdir(INPUT_FOLDER)):
        category_path = os.path.join(INPUT_FOLDER, category)
        
        if not os.path.isdir(category_path):
            continue
        
        categories_processed += 1
        
        # Nom du fichier de sortie
        output_file = os.path.join(OUTPUT_FOLDER, f"{category}_MERGED.csv")
        
        # Fusionner
        if merge_csv_files(category_path, category, output_file):
            categories_success += 1
    
    # Rapport final
    print("\n" + "=" * 70)
    print(" RAPPORT FINAL")
    print("=" * 70)
    print(f" Catégories fusionnées: {categories_success}/{categories_processed}")
    print(f"\n CSV fusionnés dans: {os.path.abspath(OUTPUT_FOLDER)}")
    
    # Lister les fichiers créés
    if os.path.exists(OUTPUT_FOLDER):
        print("\n Fichiers créés:")
        merged_files = sorted(Path(OUTPUT_FOLDER).glob("*.csv"))
        
        for mf in merged_files:
            size_kb = mf.stat().st_size / 1024
            
            # Compter les lignes
            with open(mf, 'r', encoding='utf-8') as f:
                line_count = sum(1 for _ in f) - 1  # -1 pour l'en-tête
            
            print(f"   • {mf.name:45s} {size_kb:8.1f} KB   {line_count:7,d} lignes")
    
    print("\n✨ Fusion terminée!")

if __name__ == "__main__":
    merge_all_categories()

 Dossier DATA_BULLETINS_CSV_INDIVIDUAL introuvable!
